# Notebook 13 — Pedagogy Overlay Walkthrough (Change C: pedagogy-overlay-renderer-v1)

Marimo walkthrough of the NCCE pedagogy-overlay pipeline. For every
BIEP learning graph, we tag each cell with the one or more NCCE
pedagogy principles that apply (e.g. "PRIMM" for a code-tracing cell,
"lead_with_concepts" for a new-idea introduction).

Pipeline stages:

1. Extract the 12 NCCE pedagogy principles from
   `pedagogy_principles.pdf` via BAML `ExtractPedagogyPrinciples`
   (Change A — `baml_extracts/learning_graph.baml`).
2. Cache the 12 principles to **disk** (sha256-keyed JSON) +
   **Cognee** dataset `gh_cognee_pedagogy_dataset` (semantic-search
   fallback) — see
   [`cocoindex_flows/uk_ncce/pedagogy_cache.py`](../../cocoindex_flows/uk_ncce/pedagogy_cache.py).
3. Apply the cached principles to every cell in the learning graph
   via BAML `ApplyPedagogyPrinciples` (Change C — added to
   `baml_extracts/learning_graph.baml`).
4. Render the coloured SVG via the Pedagogy overlay tab in the
   `gemini_hackathon_gradio/an_learning_graph/` studio.

Powers the
[`2026-08-31-pedagogy-overlay-renderer-v1`](../../openspec/changes/2026-08-31-pedagogy-overlay-renderer-v1/proposal.md)
change.

In [1]:
# 1. Build / hit the pedagogy-principles cache.
import os
import pathlib

# Ensure CWD is the repo root so the relative `data/bi_ep/...` paths resolve
# regardless of where the Jupyter kernel was launched from.
_REPO_ROOT = (
    pathlib.Path(__file__).resolve().parent.parent if "__file__" in dir() else pathlib.Path.cwd()
)
for _candidate in (
    _REPO_ROOT,
    pathlib.Path("/Users/cianmacandeisigh/dev/gemini_hackathon"),
    pathlib.Path.cwd(),
):
    if (_candidate / "data").exists():
        os.chdir(_candidate)
        break
from cocoindex_flows.uk_ncce.pedagogy_cache import build_pedagogy_cache

stats = build_pedagogy_cache()
print("pedagogy_cache.stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")
print()
if stats.get("from_cache"):
    print("(Cache HIT — re-runs are O(1) when pedagogy_principles.pdf hasn't changed.)")
elif stats.get("extracted"):
    print(f"(Cache MISS — fresh extract written {stats['n_principles']} principles.)")
    print("(Cognee uploaded: " + str(stats.get("cognee_uploaded")) + ")")
else:
    print("(PDF missing — `python -m dlt_pipelines.pdf_downloader` first.)")


2026-08-31T13:29:58.596422 [info     ] Deleted old log file: /Users/cianmacandeisigh/.cognee/logs/2026-08-29_11-28-57.log [cognee.shared.logging_utils]



2026-08-31T13:29:58.596927 [info     ] Log file created at: /Users/cianmacandeisigh/.cognee/logs/2026-08-31_14-29-58.log [cognee.shared.logging_utils] log_file=/Users/cianmacandeisigh/.cognee/logs/2026-08-31_14-29-58.log



2026-08-31T13:29:58.597305 [warning  ] Cognee 1.0 changes: New API — remember/recall/forget/improve (V1 add/cognify/search still work). Session memory enabled by default (CACHING=false to disable). Multi-user access control on by default (ENABLE_BACKEND_ACCESS_CONTROL=false to disable). Agents (@cognee.agent) auto-verified on registration. See https://docs.cognee.ai/ [cognee.shared.logging_utils]



2026-08-31T13:29:58.597708 [info     ] Logging initialized            [cognee.shared.logging_utils] cognee_version=1.5.3 database_path=/Users/cianmacandeisigh/dev/gemini_hackathon/.venv/lib/python3.12/site-packages/cognee/.cognee_system/databases os_info='Darwin 25.5.0 (Darwin Kernel Version 25.5.0: Mon Apr 27 20:41:15 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6041)' python_version=3.12.13 structlog_version=25.5.0



2026-08-31T13:29:58.598138 [info     ] Database storage: /Users/cianmacandeisigh/dev/gemini_hackathon/.venv/lib/python3.12/site-packages/cognee/.cognee_system/databases [cognee.shared.logging_utils]



2026-08-31T13:29:58.909215 [info     ] auth posture: authentication=required, multi_tenant=enabled (default (no env vars set)) [get_authenticated_user]



pedagogy_cache.cognee_failed dataset=gh_cognee_pedagogy_dataset reason=asyncio.run() cannot be called from a running event loop


/Users/cianmacandeisigh/dev/gemini_hackathon/cocoindex_flows/uk_ncce/pedagogy_cache.py:395: RuntimeWarning: coroutine '_cognee_upload.<locals>._upload_all' was never awaited
  return False

pedagogy_cache.fresh_extract sha=d2c46712b830 n=12 elapsed_ms=4379


pedagogy_cache.stats:
  extracted: True
  from_cache: False
  n_principles: 12
  source_pdf_sha256: d2c46712b830
  cognee_uploaded: False
  source: live_pdf

(Cache MISS — fresh extract written 12 principles.)
(Cognee uploaded: False)


In [2]:
# 2. Inspect the 12 cached principles.
import json
from pathlib import Path

# The cache may have been written by a different CWD — search for it.
_candidates = [
    Path("data/bi_ep/syllabi_md/uk_ncce/pedagogy_principles.json"),
    Path(
        "/Users/cianmacandeisigh/dev/gemini_hackathon/data/bi_ep/syllabi_md/uk_ncce/pedagogy_principles.json"
    ),
]
CACHE_PATH = next((p for p in _candidates if p.exists()), _candidates[0])
if CACHE_PATH.exists():
    payload = json.loads(CACHE_PATH.read_text())
    principles = payload.get("principles", [])
    print(f"sha256: {payload.get('source_pdf_sha256', '<unknown>')[:16]}…")
    print(f"fetched_at: {payload.get('fetched_at', '<unknown>')}")
    print(f"source: {payload.get('source', '<unknown>')}")
    print(f"count: {len(principles)}")
    print()
    for p in principles:
        print(f"  {p['id']:<25}{p['name']}")
        print(f"    {p['summary']}")
else:
    print("(No cache yet — re-run cell 1.)")

sha256: d2c46712b830f490…
fetched_at: 2026-08-31T13:29:58Z
source: live_pdf
count: 12

  primm                    PRIMM
    Predict → Run → Investigate → Modify → Make
  pair_programming         Pair programming
    Two students at one keyboard, driver + navigator
  semantic_waves           Semantic waves
    Cycle between concrete examples and abstract notation
  lead_with_concepts       Lead with concepts
    Introduce a new idea using concepts students already know
  live_coding              Live coding
    Teacher writes code in front of the class
  worked_examples          Worked examples
    Show a fully worked solution before students attempt
  formative_assessment     Formative assessment
    Mini-checks for understanding after every 10-15 min
  talking_points           Talking points
    Structured partner-talk prompts to surface thinking
  unplugged_first          Unplugged first
    Introduce concepts without a computer first
  spaced_retrieval         Spaced retrieval
    R

In [3]:
# 3. Re-run for an O(1) hit (sha256 unchanged).
stats2 = build_pedagogy_cache()
print(
    f"Re-run stats: extracted={stats2['extracted']}, from_cache={stats2['from_cache']}, n_principles={stats2['n_principles']}"
)
assert stats2["from_cache"], "Expected cache hit when PDF sha256 unchanged"
print("✓ Cache hit confirmed.")


pedagogy_cache.hit sha=d2c46712b830 n=12 elapsed_ms=0


Re-run stats: extracted=False, from_cache=True, n_principles=12
✓ Cache hit confirmed.


In [4]:
# 4. Apply the overlay to the NCCE Y8 Python learning graph.
try:
    from baml_client import b
    from baml_client.types import LearningGraph

    BAML_AVAILABLE = True
except ImportError:
    BAML_AVAILABLE = False
    b = None
    LearningGraph = None
    print("baml_client not generated — skipping live overlay application.")

if BAML_AVAILABLE:
    # In production this graph comes from the Firestore `learningGraphs/{id}`
    # document — populated by the `uk_ncce_learning_graphs` Dagster asset group.
    # For the demo we construct a minimal LearningGraph with 3 cells.
    demo_graph = LearningGraph(
        id="uk_ncce_y8_intro_to_python",
        jurisdiction="United Kingdom (NCCE)",
        subject="computer_science",
        year_level=8,
        rows=[],
        columns=[],
        cells=[],
        prerequisite_edges=[],
        pedagogy_principle_ids=[p["id"] for p in principles],
        skill_ribbons=[],
        source_pdf="data/bi_ep/syllabi_raw/uk_ncce/curriculum/learning_graph_intro_to_python_programming_y8.pdf",
        source_pages=[1, 2, 3],
        generated_at="2026-08-31T00:00:00Z",
    )
    print(f"(Live BAML overlay — graph has {len(demo_graph.cells)} cells.)")
else:
    print(
        "(Demo graph construction deferred to the production DAG asset — see orchestration/defs/3_model_lifecycle/pedagogy_overlay.py)"
    )
    print("(Live run requires Change A's `learning_graph.baml` + the BAML client runtime.)")
    print(
        "(Materialise the 6 annotated graphs via `python scripts/materialise_annotated_learning_graphs.py` for the dev path.)"
    )

baml_client not generated — skipping live overlay application.
(Demo graph construction deferred to the production DAG asset — see orchestration/defs/3_model_lifecycle/pedagogy_overlay.py)
(Live run requires Change A's `learning_graph.baml` + the BAML client runtime.)
(Materialise the 6 annotated graphs via `python scripts/materialise_annotated_learning_graphs.py` for the dev path.)


In [5]:
# 5. (Optional) Filter by principle — "show only 'Lead with concepts' cells".
if CACHE_PATH.exists() and principles:
    target_principle = "lead_with_concepts"
    matching_principle = next(
        (p for p in principles if p["id"] == target_principle),
        None,
    )
    if matching_principle:
        print(f"Filter: '{matching_principle['name']}'")
        print(f"  Summary: {matching_principle['summary']}")
        print(f"  How to apply: {matching_principle['how_to_apply']}")
        print()
        print("In the Gradio Pedagogy tab, only cells whose `cell_annotations[cell_id]`")
        print("contains this principle's id will be fully visible. Other cells are")
        print("greyed out. Click a visible cell to see the principle hover-card with the")
        print("text above + how_to_apply tips.")

Filter: 'Lead with concepts'
  Summary: Introduce a new idea using concepts students already know
  How to apply: Anchor new CS concepts in mathematics or everyday language; build up to formal syntax.

In the Gradio Pedagogy tab, only cells whose `cell_annotations[cell_id]`
contains this principle's id will be fully visible. Other cells are
greyed out. Click a visible cell to see the principle hover-card with the
text above + how_to_apply tips.


## Summary

- The 12 NCCE pedagogy principles are dynamically extracted from
  `pedagogy_principles.pdf` and cached to **disk** + **Cognee**,
  keyed on `sha256(pdf)`.
- A second run is an O(1) cache hit; a PDF change triggers a fresh
  extract automatically.
- The BAML `ApplyPedagogyPrinciples` function annotates every cell
  in the learning graph with the principle IDs that apply.
- The Pedagogy overlay tab in the Gradio studio renders the
  coloured SVG + hover-cards + principle filter dropdown.

See [`proposal.md`](../../openspec/changes/2026-08-31-pedagogy-overlay-renderer-v1/proposal.md)
for the full Phase 1-5 plan.